# Module 07: Weather Ground-Stop Classifier

In [ ]:
import pandas as pd
import requests
import io
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

print("Fetching LIVE METAR from Iowa Environmental Mesonet for VABO (Vadodara)...")
url = "https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py?station=VABO&data=tmpf,dwpf,sknt,vsby&year1=2023&month1=1&day1=1&year2=2024&month2=1&day2=1&tz=Etc%2FUTC&format=onlycomma&latlon=no&elev=no&missing=M&trace=T&direct=no&report_type=1&report_type=2"
res = requests.get(url)
df = pd.read_csv(io.StringIO(res.content.decode('utf-8'))).dropna()

df.columns = ['station', 'valid', 'temp_f', 'dew_f', 'wind_kt', 'vis_miles']
df['vis_miles'] = pd.to_numeric(df['vis_miles'], errors='coerce')
df = df.dropna()

df['ground_stop_risk'] = (df['vis_miles'] < 1.0).astype(int)
X = df[['temp_f', 'dew_f', 'wind_kt', 'vis_miles']]
y = df['ground_stop_risk']

if len(y) > 10:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    clf = RandomForestClassifier(n_estimators=50)
    clf.fit(X_train, y_train)
    print(f"Weather Model Accuracy: {clf.score(X_test, y_test):.2f}")
    joblib.dump(clf, '07_weather_rf.pkl')
